# Day 3: CHB-MIT seizure detection from raw EEG
Runtime > Change runtime type > **T4 GPU**, then run the cells in order. Expect roughly 1.5-3 hours in total; the window cache is written per case, so a disconnect can be resumed by re-running the same cell.

In [ ]:
import torch; print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
!pip -q install mne awscli joblib
!git clone -q https://github.com/aryan9-6-5/MedTech-Suite.git
%cd MedTech-Suite/day-3-chbmit-seizure-detection
!mkdir -p data results

In [ ]:
# CHB-MIT from PhysioNet's public S3 mirror: EDF files + per-case summaries (~42 GB)
!df -h /content | tail -1
!aws s3 sync --no-sign-request --only-show-errors s3://physionet-open/chbmit/1.0.0 data/chbmit --exclude "*" --include "*.edf" --include "*-summary.txt" --include "RECORDS"
!df -h /content | tail -1

In [ ]:
# sanity check: every file in RECORDS should be on disk
from pathlib import Path
rec=[l.strip() for l in open('data/chbmit/RECORDS') if l.strip()]
missing=[r for r in rec if not Path('data/chbmit',r).exists()]
print(len(rec),'records listed;',len(missing),'missing', missing[:5])

In [ ]:
%cd src
# 18-channel bipolar montage, 0.5-45 Hz, 4 s windows. Cached per case (resumable).
!python cache.py

In [ ]:
# 1D CNN on raw windows, patient-level split (held-out patients never seen in training)
!python train.py

In [ ]:
!python baseline.py

In [ ]:
# score every validation and test recording end to end (no sampling)
!python score.py

In [ ]:
!python evaluate.py

### Leakage comparison
Same CNN and baseline, but windows split at random across all patients (neighbouring windows from one recording land on both sides).

In [ ]:
!SPLIT=random python train.py
!SPLIT=random python baseline.py

In [ ]:
%cd ..
import json, glob
from IPython.display import Image, display
for f in sorted(glob.glob('results/windows_*.json')):
    m=json.load(open(f)); m.pop('history',None); print(f, json.dumps(m, indent=1))
print(open('results/events.json').read())
for f in ('sens_vs_fa','timeline'): display(Image(f'results/{f}.png'))